In [3]:
# !pip install langchain langchain-core langchain-community pypdf pymupdf sentence-transformers chromadb

In [1]:
from langchain_core.documents import Document

In [3]:
sample_doc = Document(
    page_content="Hello World!",
    metadata={"source": "https://www.google.com"}
)

In [4]:
sample_doc

Document(metadata={'source': 'https://www.google.com'}, page_content='Hello World!')

In [5]:
type(sample_doc)

langchain_core.documents.base.Document

In [6]:
# Text data
from langchain_community.document_loaders.text import TextLoader

loader = TextLoader("./python.txt", encoding="utf-8")

/var/folders/vr/kt9ts1dj10968jpzwm6zqr2m0000gn/T/ipykernel_4559/3705138437.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders.text import TextLoader


In [7]:
document = loader.load()

In [8]:
document

[Document(metadata={'source': './python.txt'}, page_content='Python is a high-level, interpreted programming language that has become one of the most popular and widely used languages in the world. Created by Guido van Rossum and first released in 1991, Python emphasizes simplicity and readability, making it easy for beginners to learn while remaining powerful for experienced developers. Its clean and concise syntax allows programmers to write fewer lines of code compared to many other languages, enhancing productivity and maintainability. Python supports multiple programming paradigms, including procedural, object-oriented, and functional programming, which makes it versatile for a wide range of applications.\\nSome key features and benefits of Python include:\\n* Ease of Learning: Simple syntax and readability make Python beginner-friendly.\\n* Versatility: Suitable for web development, data analysis, artificial intelligence, machine learning, scientific computing, automation, and mo

In [9]:
# # PDF data
# from langchain_community.document_loaders.pdf import PyPDFLoader

# pdf_loader = PyPDFLoader("data/research.pdf")

# document = pdf_loader.load()
# document

In [10]:
# # PDF data
# from langchain_community.document_loaders.pdf import PyMuPDFLoader

# pdf_loader = PyMuPDFLoader("data/research.pdf")

# document = pdf_loader.load()
# document

## Ingestion Pipeline

In [11]:
# Data => Documents
import os
from langchain_community.document_loaders.pdf import PyPDFLoader

### Documents

In [12]:
def load_all_pdfs():
    folder_path = "data"
    num_docs = 0
    all_docs = []

    for filename in os.listdir(folder_path):
        if filename.lower().endswith(".pdf"):
            # complete file path
            pdf_path = os.path.join(folder_path, filename)

            loader = PyPDFLoader(pdf_path)
            doc = loader.load()
            
            all_docs.extend(doc)
            num_docs += 1

    print("total pdfs:", num_docs)
    print("total pages:", len(all_docs))
    return all_docs

In [13]:
all_pdf_documents = load_all_pdfs()

total pdfs: 1
total pages: 21


In [24]:
type(all_pdf_documents[1])

langchain_core.documents.base.Document

### Chunks

In [14]:
# chunks
# !pip install langchain_text_splitters

In [15]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

def split_docs(documents, chunk_size=500, chunk_overlap=50):
    
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size = chunk_size,
        chunk_overlap = chunk_overlap
    )

    chunked_docs = text_splitter.split_documents(documents)
    return chunked_docs

In [17]:
chunks = split_docs(all_pdf_documents)

In [18]:
len(chunks)

244

### Embedding

In [19]:
from sentence_transformers import SentenceTransformer

In [20]:
class EmbeddingManager:
    def __init__(self, model_name="all-MiniLM-L6-v2"):
        
        self.model_name=model_name
        print("loading model....", self.model_name)
        self.model = SentenceTransformer(self.model_name)
        print("embedding dimensions=", self.model.get_sentence_embedding_dimension())


    def generate_embeddings(self, text):
        embeddings = self.model.encode(text, show_progress_bar=True)
        print("embeddings shape:", embeddings.shape)
        return embeddings

In [21]:
embedding_manager = EmbeddingManager()

loading model.... all-MiniLM-L6-v2


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

embedding dimensions= 384


/var/folders/vr/kt9ts1dj10968jpzwm6zqr2m0000gn/T/ipykernel_4559/4021223045.py:7: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("embedding dimensions=", self.model.get_sentence_embedding_dimension())


### Vector Store

In [22]:
import chromadb
import uuid

In [23]:
class VectorStoreManager:
    def __init__(self, persist_directory="data/vector_store", collection_name="pdf_documents"):
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.collection = None
        self.client = None

        self._initialize_store()

    def _initialize_store(self):
        os.makedirs(self.persist_directory, exist_ok=True)
        
        # create a client
        self.client = chromadb.PersistentClient(path=self.persist_directory)

        # create the collection
        self.collection = self.client.get_or_create_collection(
            name=self.collection_name,
            metadata={"description": "vector store collection for pdf embeddings in RAG"}
        )

        print("initialized the vector store with collection:", self.collection_name)
        print("docs in collection:", self.collection.count())

    def add_documents(self, documents, embeddings):
        if len(documents) != len(embeddings):
            raise ValueError("num of documents does not match num of embeddings")


        # store => ids, embedding, document, metadata
        ids = []
        all_metadata = []
        documents_content = []
        embeddings_list = []

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            doc_id = f"doc_{uuid.uuid4()}"
            ids.append(doc_id)

            metadata = dict(doc.metadata)
            metadata["doc_index"] = i
            metadata["content_length"] = len(doc.page_content)
            all_metadata.append(metadata)

            documents_content.append(doc.page_content)

            embeddings_list.append(embedding.tolist())

            self.collection.add(
                ids=ids,
                metadatas=all_metadata,
                documents=documents_content,
                embeddings=embeddings_list
            )

        print("total documents added in vector store=", len(documents_content))
        print("docs in collection:", self.collection.count())

In [24]:
vector_store = VectorStoreManager()

initialized the vector store with collection: pdf_documents
docs in collection: 0


In [25]:
# data => documents => chunks => embeddings => store in vector store

texts = [doc.page_content for doc in chunks]

emebedding = embedding_manager.generate_embeddings(texts)

vector_store.add_documents(chunks, emebedding)

Batches:   0%|          | 0/8 [00:00<?, ?it/s]

embeddings shape: (244, 384)
total documents added in vector store= 244
docs in collection: 244


# Retrieval Pipeline

In [26]:
from sklearn.metrics.pairwise import cosine_similarity

In [27]:
class RAGRetriever:
    def __init__(self, embedding_manager, vector_store):
        self.embedding_manager = embedding_manager
        self.vector_store = vector_store


    def retrieve(self, query, top_k=5, score_threshold=0.0):
        # query => embedding
        query_embeddings = self.embedding_manager.generate_embeddings([query])[0]

        # semantic search
        results = self.vector_store.collection.query(
            query_embeddings=[query_embeddings.tolist()],
            n_results=top_k
        )

        # cosine similarity
        retrieved_docs=[]
        
        if results["documents"] and results["documents"][0]:
            ids = results["ids"][0]
            metadatas = results["metadatas"][0]
            documents = results["documents"][0]
            distances = results["distances"][0]

            for i, (doc_id, metadata, document, distance) in enumerate(zip(ids, metadatas, documents, distances)):
                similarity_score = 1 - distance

                if similarity_score >= score_threshold:
                    retrieved_docs.append({
                        "id": doc_id,
                        "document": document,
                        "metadata": metadata,
                        "distance": distance,
                        "similarity_score": similarity_score,
                        "rank" : i + 1
                    })

            print(f"retrieved {len(retrieved_docs)} documents")

        else:
            print("no documents found")

        return retrieved_docs

In [28]:
rag_retriever = RAGRetriever(embedding_manager, vector_store)

In [30]:
rag_retriever.retrieve("What is RAG ?")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

embeddings shape: (1, 384)
retrieved 5 documents


[{'id': 'doc_d5c00037-109c-4ee7-8bc8-81a8f8780698',
  'document': 'and speculate on upcoming trends and innovations.\nOur contributions are as follows:\n• In this survey, we present a thorough and systematic\nreview of the state-of-the-art RAG methods, delineating\nits evolution through paradigms including naive RAG,\narXiv:2312.10997v5  [cs.CL]  27 Mar 2024',
  'metadata': {'trapped': '/False',
   'content_length': 288,
   'title': '',
   'subject': '',
   'source': 'data/Research.pdf',
   'moddate': '2024-03-28T00:54:45+00:00',
   'total_pages': 21,
   'author': '',
   'producer': 'pdfTeX-1.40.25',
   'page_label': '1',
   'creator': 'LaTeX with hyperref',
   'page': 0,
   'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5',
   'creationdate': '2024-03-28T00:54:45+00:00',
   'doc_index': 11,
   'keywords': ''},
  'distance': 0.46291494369506836,
  'similarity_score': 0.5370850563049316,
  'rank': 1},
 {'id': 'doc_40f8f959-ca97-4

# Integrate with LLMs

## OpenAI - GPT

In [61]:
API_KEY_OPENAI = "paste-your-api-key-here"

In [63]:
# !pip install langchain-openai

In [64]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    openai_api_key=API_KEY_OPENAI,
    model="gpt-5.4",
    temperature=0.1,
    max_tokens=1024
)

In [75]:
# generate our retrieval-augmented output
def generate_output(query, retriever, llm, top_k=3):
    results = retriever.retrieve(query, top_k)

    context = "\n".join([doc["document"] for doc in results]) if results else ""

    if not context:
        print("we found no relevant context for the given query")

    # context + query
    prompt = f""" use given context to generate the answer for the query
                Context: {context}
                Query: {query} """

    response = llm.invoke(prompt) # expecting a string as prompt
    return response.content

In [79]:
answer = generate_output("what is encoder-decoder?", rag_retriever, llm)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

embeddings shape: (1, 384)
retrieved 3 documents


In [80]:
print(answer)

An encoder-decoder is a neural network architecture with two main parts:

- **Encoder**: processes the input and converts it into internal representations.
- **Decoder**: takes the encoder’s output and generates the target output.

From the given context:

- The **encoder** is a stack of **N = 6 identical layers**.
- Each encoder layer has **two sub-layers**:
  1. **multi-head self-attention**
  2. a **position-wise feed-forward network**

- The **decoder** is also a stack of **N = 6 identical layers**.
- Each decoder layer has the same kinds of sub-layers as the encoder, but adds a **third sub-layer** that performs **multi-head attention over the output of the encoder stack**.

So, **encoder-decoder** means a model where:
1. the **encoder reads and represents the input**, and
2. the **decoder uses that representation to produce the output**.


### Groq

In [81]:
API_Key_GROQ = "paste-your-api-key-here"

In [83]:
# !pip install langchain-groq

In [84]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    groq_api_key=API_Key_GROQ,
    model="qwen/qwen3-32b",
    temperature=0.1,
    max_tokens=1024
)

In [85]:
# generate our retrieval-augmented output
def generate_output(query, retriever, llm, top_k=3):
    results = retriever.retrieve(query, top_k)

    context = "\n".join([doc["document"] for doc in results]) if results else ""

    if not context:
        print("we found no relevant context for the given query")

    # context + query
    prompt = f""" use given context to generate the answer for the query
                Context: {context}
                Query: {query} """

    response = llm.invoke([prompt.format(context=context, query=query)]) # expecting a list as prompt
    return response.content

In [88]:
answer = generate_output("what is RAG?", rag_retriever, llm)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

embeddings shape: (1, 384)
retrieved 3 documents


In [89]:
print(answer)

<think>
Okay, the user is asking "what is RAG?" and I need to use the provided context to answer. Let me start by reading through the context carefully.

The context mentions that the paper is a survey on RAG methods, discussing their evolution through paradigms like naive RAG and effective RAG. It also talks about evaluation methods, tasks, datasets, and future directions. The paper is structured with sections on main concepts, paradigms, and integration with LLMs.

So, RAG stands for Retrieval-Augmented Generation. From the context, it's a method that combines retrieval of information with generation, likely using large language models. The survey covers different paradigms of RAG, like naive and effective frameworks. The key points to include are the definition, purpose (integrating retrieval with generation), and maybe the evolution mentioned in the context. Also, the context mentions that RAG is used within LLMs, so that's important. I should avoid technical jargon but still be pr

In [91]:
# Claude
# !pip install langchain-anthropic